In [14]:
import numpy as np
import pandas as pd
import keras
from keras import layers, ops
from sklearn.model_selection import train_test_split

In [ ]:
def load_data_and_encode(dataset_path, nuc_to_idx, max_rows):
    """
    Read the Ribonanza CSV file, keep the sequences measured with both
    experiments (DMS_MaP and 2A3_MaP) and encode each nucleotide sequence
    as a list of integer indices.

    Parameters
    ----------
    dataset_path : str
        Path to the CSV file (train_data_QUICK_START.csv). Each sequence
        appears on two rows, one per experiment_type.
    nuc_to_idx : dict
        Mapping from nucleotide character (A/C/G/U) to integer index.
    max_rows : int or None
        Number of sequences to keep (useful to test quickly).
        None keeps all the sequences.

    Returns
    -------
    df_dms : pd.DataFrame
        Rows of the DMS_MaP experiment, sorted by sequence_id.
    df_2a3 : pd.DataFrame
        Rows of the 2A3_MaP experiment, sorted by sequence_id.
        Row i of df_dms and row i of df_2a3 are the same sequence.
    sequences : list of str
        Nucleotide sequences as strings.
    encoded_seqs : list of list of int
        Nucleotide sequences encoded as lists of integer indices.
    """
    sequence_dataset = pd.read_csv(dataset_path)

    # We split the two experiments in two tables
    df_dms = sequence_dataset[sequence_dataset["experiment_type"] == "DMS_MaP"]
    df_2a3 = sequence_dataset[sequence_dataset["experiment_type"] == "2A3_MaP"]

    # We keep only the sequences which have the two measures
    ids_dms = set(df_dms["sequence_id"])
    ids_2a3 = set(df_2a3["sequence_id"])
    common_ids = sorted(ids_dms & ids_2a3)

    if max_rows is not None:
        common_ids = common_ids[:max_rows]

    # We sort by sequence_id so that the two tables are in the same order
    df_dms = df_dms[df_dms["sequence_id"].isin(common_ids)]
    df_dms = df_dms.drop_duplicates(subset="sequence_id")
    df_dms = df_dms.sort_values("sequence_id")

    df_2a3 = df_2a3[df_2a3["sequence_id"].isin(common_ids)]
    df_2a3 = df_2a3.drop_duplicates(subset="sequence_id")
    df_2a3 = df_2a3.sort_values("sequence_id")

    # The sequences are the same in the two tables
    sequences = df_dms["sequence"].astype(str).tolist()

    # We want to turn letters into numbers before putting them into a vector
    encoded_seqs = []

    # We map each letter to its number
    for seq in sequences:
        encoded_seq = []

        for c in seq:
            encoded_seq.append(nuc_to_idx.get(c, 0))

        encoded_seqs.append(encoded_seq)

    return df_dms, df_2a3, sequences, encoded_seqs

In [ ]:
def prepare_data(encoded_seqs, y, maxlen):
    """
    Pad the encoded sequences to a fixed length, then split the dataset
    into training and test sets.

    Parameters
    ----------
    encoded_seqs : list of list of int
        Nucleotide sequences encoded as lists of integer indices.
    y : np.ndarray of shape (n_samples, maxlen, 2)
        Targets (output of build_targets).
    maxlen : int
        Fixed length to which every sequence is padded or truncated.

    Returns
    -------
    X : np.ndarray of shape (n_samples, maxlen)
        Padded encoded sequences.
    x_train, x_test : np.ndarray
        Train/test split of X.
    y_train, y_test : np.ndarray
        Train/test split of y.
    """
    X = keras.utils.pad_sequences(
        encoded_seqs,
        maxlen=maxlen,
        padding="post",
        truncating="post",
        value=0,
    )

    x_train, x_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    return X, x_train, x_test, y_train, y_test

In [15]:
VOCAB = {"A": 1, "C": 2, "G": 3, "U": 4, "EMPTY": 0}

In [17]:
def fill_empty(list_of_sequences):
    number_of_sequence = len(list_of_sequences)
    seq_length_max = 0
    for sequence in list_of_sequences:
        length = len(sequence)
        if length > seq_length_max:
            seq_length_max = length
            
    tokens = np.full((number_of_sequence, seq_length_max), VOCAB["EMPTY"])
    
    for line, seq in enumerate(list_of_sequences):
        tokens[line,0 :len(seq)] = seq
    mask = tokens != VOCAB["EMPTY"]
    return tokens, mask

In [18]:
class NucleotideEmbedding(layers.Layer):
    def __init__(self, dim=192, **kwargs):
        super().__init__(**kwargs)
        self.emb = layers.Embedding(len(VOCAB), dim, mask_zero=True)

    def call(self, tokens):
        return self.emb(tokens)

In [30]:
import numpy as np

def create_bpp_matrix(bpp_file, size):
    grille = np.zeros((size, size))

    with open(bpp_file) as f:
        for ligne in f:
            if not ligne.strip():
                continue

            i, j, proba = ligne.split()
            i, j = int(i) - 1, int(j) - 1

            grille[i, j] = float(proba)
            grille[j, i] = float(proba)

    return grille


In [ ]:
def build_mini_model(dim, n_outputs):
    tokens = keras.Input(shape=(None,), dtype="int32") 
    x = NucleotideEmbedding(dim)(tokens)  
    x = layers.Conv1D(dim, kernel_size=3, padding="same", activation="relu")(x)
    x = layers.Dense(n_outputs)(x)  
    if n_outputs == 1:
        x = layers.Lambda(lambda t: ops.squeeze(t, axis=-1))(x) 
    return keras.Model(tokens, x, name="mini_model")


In [ ]:
train = pd.read_csv("train.csv")
print(train.head())
print(train.columns)